# Pertemuan 11 — Dynamic Programming II
LCS, Edit Distance, Floyd–Warshall, dan **[Modern]** DP untuk sekuens: Viterbi (HMM), Dynamic Time Warping.

In [ ]:
import math, random, time
import numpy as np
random.seed(11)

## 1. Longest Common Subsequence — O(mn) waktu

In [ ]:
def lcs(a, b):
    m, n = len(a), len(b); dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            dp[i+1][j+1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j+1], dp[i+1][j])
    out, i, j = [], m, n
    while i and j:
        if a[i-1] == b[j-1]: out.append(a[i-1]); i -= 1; j -= 1
        elif dp[i-1][j] >= dp[i][j-1]: i -= 1
        else: j -= 1
    return dp[m][n], "".join(reversed(out))
print(lcs("AGGTAB", "GXTXAYB"), lcs("kompleksitas", "komputasi"))

## 2. Edit Distance + rekonstruksi operasi (dasar spell checker dan `diff`)

In [ ]:
def edit_distance(a, b):
    m, n = len(a), len(b); dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + (a[i-1] != b[j-1]))
    ops, i, j = [], m, n
    while i or j:
        if i and j and dp[i][j] == dp[i-1][j-1] + (a[i-1] != b[j-1]):
            ops.append("cocok" if a[i-1] == b[j-1] else f"ganti {a[i-1]}→{b[j-1]}"); i -= 1; j -= 1
        elif i and dp[i][j] == dp[i-1][j] + 1: ops.append(f"hapus {a[i-1]}"); i -= 1
        else: ops.append(f"sisip {b[j-1]}"); j -= 1
    return dp[m][n], list(reversed(ops))
d, ops = edit_distance("kitten", "sitting"); print(d, ops)

kamus = ["algoritma", "kompleksitas", "rekursi", "greedy", "program", "dinamis", "backtracking", "optimal"]
def koreksi(kata, k=3): return sorted(kamus, key=lambda w: edit_distance(kata, w)[0])[:k]
print("koreksi('algortima') →", koreksi("algortima"), "| koreksi('kompleksitsa') →", koreksi("kompleksitsa"))

## 3. Floyd–Warshall — O(V³) semua pasangan

In [ ]:
def floyd(W):
    n = len(W); d = [row[:] for row in W]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if d[i][k] + d[k][j] < d[i][j]: d[i][j] = d[i][k] + d[k][j]
    return d
INF = math.inf
W = [[0, 3, INF, 7], [8, 0, 2, INF], [5, INF, 0, 1], [2, INF, INF, 0]]
for row in floyd(W): print(row)

## 4. [Modern] Viterbi untuk HMM — O(T·S²)
Contoh klasik: keadaan tersembunyi {Sehat, Demam}, pengamatan {normal, dingin, pusing}.

In [ ]:
states = ["Sehat", "Demam"]
start = {"Sehat": .6, "Demam": .4}
trans = {"Sehat": {"Sehat": .7, "Demam": .3}, "Demam": {"Sehat": .4, "Demam": .6}}
emit  = {"Sehat": {"normal": .5, "dingin": .4, "pusing": .1}, "Demam": {"normal": .1, "dingin": .3, "pusing": .6}}

def viterbi(obs):
    V = [{s: math.log(start[s]) + math.log(emit[s][obs[0]]) for s in states}]; back = []
    for o in obs[1:]:
        row, bp = {}, {}
        for s in states:
            prev = max(states, key=lambda p: V[-1][p] + math.log(trans[p][s]))
            row[s] = V[-1][prev] + math.log(trans[prev][s]) + math.log(emit[s][o]); bp[s] = prev
        V.append(row); back.append(bp)
    last = max(states, key=lambda s: V[-1][s]); path = [last]
    for bp in reversed(back): path.append(bp[path[-1]])
    return list(reversed(path)), math.exp(V[-1][last])
print(viterbi(["normal", "dingin", "pusing"]))

Menyebutkan semua jalur = Sᵀ; Viterbi = T·S². Ukur pertumbuhannya:

In [ ]:
rng = random.Random(0)
for T in [10, 100, 1000, 10000]:
    obs = [rng.choice(["normal", "dingin", "pusing"]) for _ in range(T)]
    t = time.perf_counter(); viterbi(obs); print(f"T={T:>6}: {time.perf_counter()-t:.4f}s  (brute force: 2^{T} jalur)")

## 5. [Modern] Dynamic Time Warping — O(mn)

In [ ]:
def dtw(a, b):
    m, n = len(a), len(b); D = np.full((m + 1, n + 1), np.inf); D[0, 0] = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            D[i, j] = abs(a[i-1] - b[j-1]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return D[m, n]
t = np.linspace(0, 2 * np.pi, 60)
s1, s2, s3 = np.sin(t), np.sin(t * 1.0 + 0.0)[::1] * 1.0, np.sin(np.linspace(0, 2 * np.pi, 90))
print("DTW(sin, sin sama panjang) =", round(dtw(s1, s2), 3), "| DTW(sin 60 titik, sin 90 titik) =", round(dtw(s1, s3), 3),
      "| Euclid tidak bisa dipakai pada panjang berbeda")

## 6. Latihan / Mini Proyek (pilih satu)
1. **Matrix Chain Multiplication** — urutan kurung optimal, O(n³); bandingkan jumlah perkalian dengan urutan naif.
2. **Sequence alignment** (Needleman–Wunsch) untuk DNA dengan skor match=+1, mismatch=−1, gap=−2.
3. **Spell checker** dengan kamus 10.000 kata; ukur waktu dan percepat dengan batas jarak maksimum (*banded* edit distance).
4. **Bellman–Ford** dan deteksi siklus negatif; bandingkan dengan Dijkstra.

In [ ]:
def matrix_chain(dims):
    pass  # TODO